In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

In [25]:
from docling.document_converter import DocumentConverter
from pathlib import Path

converter = DocumentConverter()
raw_path = Path("data") / "raw" / "Deep learning-based approach to diagnose lung cancer using CT-scan images.pdf"
result = converter.convert(raw_path)


[INFO] 2026-09-22 22:59:21,229 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-22 22:59:21,313 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-22 22:59:21,879 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-22 22:59:21,882 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-22 22:59:26,240 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-22 22:59:26,247 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-22 22:59:26,328 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-22 22:59:26,345 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapido

In [ ]:
for item, _ in result.document.iterate_items():
    label = getattr(item, "label", None)
    text = getattr(item, "text", None)
    
    if label == "table":
        print("label: ", label)
        print("text: ", text)

In [9]:

for item, _ in result.document.iterate_items():
    label = getattr(item, "label", None)
    text = getattr(item, "text", None)
    
    if label == "table":
        print("label: ", label)
        print("text: ", text)
        print([a for a in dir(item) if not a.startswith("_")])


label:  table
text:  None
['add_annotation', 'annotations', 'caption_text', 'captions', 'children', 'comments', 'construct', 'content_layer', 'copy', 'data', 'dict', 'export_to_dataframe', 'export_to_doctags', 'export_to_document_tokens', 'export_to_html', 'export_to_markdown', 'export_to_otsl', 'footnotes', 'from_orm', 'get_annotations', 'get_image', 'get_location_tokens', 'get_ref', 'image', 'json', 'label', 'meta', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'parent', 'parse_file', 'parse_obj', 'parse_raw', 'prov', 'references', 'schema', 'schema_json', 'self_ref', 'source', 'update_forward_refs', 'validate']
label:  table
text:  None
['add_annotation', 'annotations', 'caption_text', 'captions', 'children', 'comments', 'co

In [10]:
print(item.caption_text(result.document))
print(item.export_to_markdown(doc=result.document))


AttributeError: 'ListItem' object has no attribute 'caption_text'

In [14]:
from docling_core.types.doc import DocItemLabel, TableItem, FormulaItem

for item, _ in result.document.iterate_items():
    if isinstance(item, TableItem) or getattr(item, "label", None) == DocItemLabel.TABLE:
        print("label", item.label)
        
        try:
            df = item.export_to_dataframe(doc=result.document)
            print(df.to_string(index=False))
        except Exception as e:
            print("error", {e})     

label table
       Dataset Number of Images Splitting percentage
  Training set              613                 70 %
   Testing set              315                 20 %
Validating set               72                 10 %
label table
                                                                     Hyperparameter                                                                               Value
Optimizer Loss function Metrics Early Stopping Patience Number of epochs Batch size Adam categorical_crossentropy accuracy, precision, recall True 20 150 (default: 32)
label table
                                                         Predictive Class Positives              Predictive Class Negative
Actual Class Positives Actual Class Negative True Positive (TP) False Positive (FP) False Negative (FN) True Negative (TN)
label table
        Models Precision (%) Recall (%) F1-Score (%) Accuracy (%)
  Enhanced CNN        99.2 %     98.0 %       98.4 %        100 %
      ConvNeXt        86.7

In [26]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.do_formula_enrichment = True

converter  = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})

for item, _ in result.document.iterate_items():
    label = getattr(item, "label", None)
    text = getattr(item, "text", None)
    
    if label == "formula":
        print("label: ", label)
        print("text: ", text)

label:  formula
text:  
label:  formula
text:  
label:  formula
text:  
label:  formula
text:  


In [20]:
for item, _ in result.document.iterate_items():
    if isinstance(item, FormulaItem) or getattr(item, "label", None) == DocItemLabel.FORMULA:
        print("label", item.label)
        
        try:
            df = item.text
            print(df)
        except Exception as e:
            print("error", {e})     

label formula

label formula

label formula

label formula



In [27]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from pathlib import Path

# 1. Configure the high-precision extraction pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.do_formula_enrichment = True
pipeline_options.do_ocr = True  # Fallback to OCR if formulas are flattened graphics
pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

# 2. Bind options to the PDF format handler
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# 3. RE-RUN THE CONVERSION (Crucial step to apply the new configuration!)
raw_path = Path("data") / "raw" / "Deep learning-based approach to diagnose lung cancer using CT-scan images.pdf"
result = converter.convert(raw_path)

# 4. Iterate and print out the freshly processed results
for item, _ in result.document.iterate_items():
    label = getattr(item, "label", None)
    
    # Check using Docling's safe string conversion
    if str(label) == "formula" or label == "formula":
        text = getattr(item, "text", None)
        print("label:", label)
        print("text:", repr(text))  # repr() will show hidden spaces or formatting clearly

[INFO] 2026-09-22 23:14:40,582 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-22 23:14:40,598 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-22 23:14:40,787 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-22 23:14:40,787 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-22 23:14:42,267 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-22 23:14:42,271 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-22 23:14:42,287 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-22 23:14:42,295 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapido

OSError: [WinError 1314] A required privilege is not held by the client: '..\\..\\blobs\\a6344aac8c09253b3b630fb776ae94478aa0275b' -> 'G:\\Projects\\hf_cache\\hub\\models--docling-project--CodeFormulaV2\\snapshots\\ecedbe111d15c2dc60bfd4a823cbe80127b58af4\\.gitattributes'

In [4]:
import sys
sys.path.insert(0, "src")
from src.extract_text import extract_blocks
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
from pathlib import Path

raw_path = Path("data") / "raw" / "Deep learning-based approach to diagnose lung cancer using CT-scan images.pdf"
result = converter.convert(raw_path)


blocks = extract_blocks(result.document)

table_blocks = [b for b in blocks if b["label"] == "table"]
print(f"Total blocks: {len(blocks)}")
print(f"Table blocks found: {len(table_blocks)}")

for b in table_blocks:
    print("---")
    print("page:", b["page"])
    print("text is blank:", not b["text"])
    print(b["text"][:200])  # first 200 chars, just to eyeball it


[INFO] 2026-09-23 00:37:07,590 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-23 00:37:07,620 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-23 00:37:07,756 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-23 00:37:07,756 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-23 00:37:09,136 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-23 00:37:09,136 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-23 00:37:09,359 [RapidOCR] download_file.py:60: File exists and is valid: G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-23 00:37:09,359 [RapidOCR] main.py:50: Using G:\Projects\agentic-ai\RAG-NEW\venv-rag-new\Lib\site-packages\rapido

Total blocks: 263
Table blocks found: 4
---
page: 9
text is blank: False
       Dataset Number of Images Splitting percentage
  Training set              613                 70 %
   Testing set              315                 20 %
Validating set               72          
---
page: 10
text is blank: False
                                                                     Hyperparameter                                                                               Value
Optimizer Loss function Metrics 
---
page: 10
text is blank: False
                                                         Predictive Class Positives              Predictive Class Negative
Actual Class Positives Actual Class Negative True Positive (TP) False Positiv
---
page: 10
text is blank: False
        Models Precision (%) Recall (%) F1-Score (%) Accuracy (%)
  Enhanced CNN        99.2 %     98.0 %       98.4 %        100 %
      ConvNeXt        86.7 %     89.5 %       88.0 %         87 %
  


In [4]:
import sys
sys.path.insert(0, "src")
from src.db import get_connection, SCHEMA_NAME

conn = get_connection()
cur = conn.cursor()
cur.execute(f"SELECT embedding FROM {SCHEMA_NAME}.text_chunks LIMIT 1")
val = cur.fetchone()[0]

print(type(val))
print([a for a in dir(val) if not a.startswith("_")])

conn.close()


<class 'pgvector.vector.Vector'>
['dimensions', 'from_binary', 'from_text', 'to_binary', 'to_list', 'to_numpy', 'to_text']


In [5]:
import sys
sys.path.insert(0, "src")
from src.db import get_connection, SCHEMA_NAME

conn = get_connection()
cur = conn.cursor()
cur.execute(f"SELECT * FROM {SCHEMA_NAME}.text_chunks LIMIT 1")
columns = [desc[0] for desc in cur.description]
row = cur.fetchone()

for col, val in zip(columns, row):
    if col == "embedding":
        vec = val.to_list()
        print(f"{col}: [vector of {len(vec)} numbers] first 5 -> {vec[:5]}")
    elif col == "text":
        print(f"{col}: {val[:200]}...")
    else:
        print(f"{col}: {val}")

conn.close()


chunk_id: 1-s2.0-S1877050925029461-main_c1
parent_id: 1-s2.0-S1877050925029461-main_p1
source_pdf: 1-s2.0-S1877050925029461-main.pdf
section: None
page_start: 1
page_end: 1
domain: lung-cancer
text: Available online at www.sciencedirect.com

Available online at www.sciencedirect.com

Available online at www.sciencedirect.com...
embedding_version: v2_bgebase_clipvitb32_structureaware
embedding: [vector of 768 numbers] first 5 -> [0.0038713780231773853, 0.010709693655371666, -0.03667071461677551, 0.04090414196252823, 0.052928272634744644]
text_search: 'avail':1,5,9 'onlin':2,6,10 'www.sciencedirect.com':4,8,12
